# DistilRoBERTa Fine-Tuning — IMDb Sentiment Analysis

**Goal:** Fine-tune `distilroberta-base` on the IMDb review dataset for binary sentiment classification.

We use HuggingFace `transformers` + `datasets` throughout:
- Pre-trained tokenizer (BPE, RoBERTa vocabulary)
- Pre-trained `distilroberta-base` model weights
- HuggingFace `Trainer` API for clean training loops
- Full evaluation suite (accuracy, precision, recall, F1, confusion matrix)
- Model download — to your computer or Google Drive

**Why DistilRoBERTa?**  
- ~40% smaller and ~60% faster than RoBERTa-base  
- Retains ~97% of RoBERTa's performance  
- Excellent accuracy–speed tradeoff for a deployed product  

**Comparison with your custom Transformer:**

| Model | Accuracy | F1 | Training data seen |
|---|---|---|---|
| Your Custom Transformer | 81.6% | 81.1% | IMDb only |
| DistilRoBERTa (expected) | ~92–94% | ~92–94% | 160GB+ pretraining corpus |

---

## Sections
1. Setup & Config
2. Dataset Loading
3. Tokenization
4. Model
5. Training
6. Evaluation
7. Inference
8. Save & Download Model

---
## 1. Setup & Config

Install and import everything we need.  
**Make sure you have a GPU runtime in Colab:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Install / upgrade dependencies (run once)
!pip install -q --upgrade datasets huggingface_hub
!pip install -q transformers accelerate scikit-learn matplotlib seaborn

In [ ]:
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

print(f"PyTorch version:    {torch.__version__}")
print(f"CUDA available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:                {torch.cuda.get_device_name(0)}")

In [ ]:
# ─── CONFIGURATION ───────────────────────────────────────────────────────────

# Set QUICK_RUN = True for a fast sanity check (2k samples, 2 epochs, ~5 min)
# Set QUICK_RUN = False for the full IMDb run (25k train, 3 epochs, ~20 min on T4)
QUICK_RUN = False

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Model
MODEL_NAME   = "distilroberta-base"
NUM_CLASSES  = 2            # positive / negative
MAX_LEN      = 512          # DistilRoBERTa max sequence length

# Training
BATCH_SIZE       = 16       # reduce to 8 if you get OOM errors
LEARNING_RATE    = 2e-5     # standard fine-tuning LR for BERT-family models
EPOCHS           = 2 if QUICK_RUN else 3
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1      # 10% of steps used for LR warmup
PATIENCE         = 2        # early stopping patience (epochs)

# Data
QUICK_SAMPLES    = 2_000    # samples per split in QUICK_RUN mode
VAL_SPLIT        = 0.15     # fraction of training set used for validation

# Output
OUTPUT_DIR       = "./distilroberta-imdb"

print(f"Model:      {MODEL_NAME}")
print(f"QUICK_RUN:  {QUICK_RUN}")
print(f"Epochs:     {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"LR:         {LEARNING_RATE}")

---
## 2. Dataset Loading

Same IMDb dataset you used in Project A — 25,000 training reviews, 25,000 test reviews.  
Labels: `0 = negative`, `1 = positive`.

In [ ]:
print("Loading IMDb dataset...")

try:
    raw_dataset = load_dataset("stanfordnlp/imdb")
except Exception as e:
    print(f"stanfordnlp/imdb failed ({e}), trying legacy path...")
    raw_dataset = load_dataset("imdb", trust_remote_code=True)

print(raw_dataset)

In [ ]:
train_full = raw_dataset["train"]
test_data  = raw_dataset["test"]

# ── QUICK_RUN: sample a balanced subset ──────────────────────────────────────
def balanced_sample(dataset, n_per_class, seed=SEED):
    """Return a balanced subset with n_per_class examples per label."""
    pos = dataset.filter(lambda x: x["label"] == 1).shuffle(seed=seed).select(range(n_per_class))
    neg = dataset.filter(lambda x: x["label"] == 0).shuffle(seed=seed).select(range(n_per_class))
    from datasets import concatenate_datasets
    return concatenate_datasets([pos, neg]).shuffle(seed=seed)

if QUICK_RUN:
    n = QUICK_SAMPLES // 2
    train_full = balanced_sample(train_full, n)
    test_data  = balanced_sample(test_data,  n // 2)

# ── Train / validation split (from training data only) ───────────────────────
split = train_full.train_test_split(test_size=VAL_SPLIT, seed=SEED)
train_data = split["train"]
val_data   = split["test"]

print(f"Train:      {len(train_data):,} examples")
print(f"Validation: {len(val_data):,} examples")
print(f"Test:       {len(test_data):,} examples")

# Quick look
sample = train_data[0]
print(f"\nSample review (first 300 chars):")
print(sample["text"][:300])
print(f"\nLabel: {'Positive' if sample['label'] == 1 else 'Negative'}")

---
## 3. Tokenization

Unlike your custom Transformer (which used a hand-built word vocabulary), DistilRoBERTa uses a **Byte-Pair Encoding (BPE) tokenizer** trained on a massive corpus.

Key differences from your Project A tokenizer:
- Handles subwords — `"fantastic"` might tokenize as `["fan", "tastic"]`
- Vocabulary of 50,265 tokens (vs your 20k word vocab)
- Adds special tokens automatically: `<s>` (start), `</s>` (end)
- Returns `attention_mask` — 1 for real tokens, 0 for padding

In [ ]:
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Vocabulary size:   {tokenizer.vocab_size:,}")
print(f"Max model length:  {tokenizer.model_max_length}")

# Demo — see how the tokenizer handles a sample review
sample_text = "This movie was absolutely fantastic! The acting was superb."
encoded = tokenizer(sample_text, max_length=32, truncation=True, padding="max_length")

print(f"\nSample: '{sample_text}'")
print(f"Token IDs:       {encoded['input_ids']}")
print(f"Attention mask:  {encoded['attention_mask']}")
print(f"Decoded tokens:  {tokenizer.convert_ids_to_tokens(encoded['input_ids'])}")

In [ ]:
def tokenize_fn(batch):
    """Tokenize a batch of reviews. Called by dataset.map()."""
    return tokenizer(
        batch["text"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )

print("Tokenizing all splits (this takes ~1-2 min on full IMDb)...")

train_tok = train_data.map(tokenize_fn, batched=True, desc="Train")
val_tok   = val_data.map(tokenize_fn,   batched=True, desc="Val")
test_tok  = test_data.map(tokenize_fn,  batched=True, desc="Test")

# Set format so the Trainer receives plain PyTorch tensors
cols = ["input_ids", "attention_mask", "label"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch",   columns=cols)
test_tok.set_format(type="torch",  columns=cols)

print("Tokenization complete.")
print(f"Features: {list(train_tok.features.keys())}")

In [ ]:
# Visualize review length distribution (in tokens, not words)
# We sample 2000 examples to keep this fast
sample_idx   = random.sample(range(len(train_data)), min(2000, len(train_data)))
sample_texts = [train_data[i]["text"] for i in sample_idx]
token_lengths = [
    len(tokenizer(t, truncation=False)["input_ids"])
    for t in sample_texts
]

plt.figure(figsize=(8, 3))
plt.hist(token_lengths, bins=50, color='steelblue', edgecolor='white')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'MAX_LEN={MAX_LEN}')
plt.xlabel("Number of tokens (BPE)")
plt.ylabel("Count")
plt.title("Review Length Distribution — BPE tokens (sample of 2,000 reviews)")
plt.legend()
plt.tight_layout()
plt.show()

pct_truncated = sum(1 for l in token_lengths if l > MAX_LEN) / len(token_lengths) * 100
print(f"Mean:         {np.mean(token_lengths):.0f} tokens")
print(f"Median:       {np.median(token_lengths):.0f} tokens")
print(f"Truncated:    {pct_truncated:.1f}% of reviews exceed MAX_LEN={MAX_LEN}")

---
## 4. Model

`AutoModelForSequenceClassification` loads `distilroberta-base` with a **two-class classification head** on top.  

Architecture:
- 6 Transformer blocks (vs 12 in RoBERTa-base)
- 768 hidden dimensions
- 82M parameters total (~40% fewer than RoBERTa-base)

Fine-tuning updates **all** weights — both the classification head and the pre-trained encoder.

In [ ]:
id2label = {0: "Negative", 1: "Positive"}
label2id = {"Negative": 0, "Positive": 1}

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
)

model = model.to(DEVICE)

# Parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

---
## 5. Training

We use HuggingFace `Trainer` — it handles:
- The training loop
- Gradient accumulation
- LR scheduling with warmup
- Checkpoint saving
- Validation at each epoch
- Early stopping

In [ ]:
def compute_metrics(eval_pred):
    """
    Called by Trainer at the end of each validation pass.
    Returns accuracy, precision, recall, and F1.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)

    return {
        "accuracy":  round(acc,  4),
        "precision": round(prec, 4),
        "recall":    round(rec,  4),
        "f1":        round(f1,   4),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Training schedule ────────────────────────────────────────────────────
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,  # larger batch for eval (no grad)
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="linear",         # linear decay after warmup

    # ── Evaluation & checkpointing ───────────────────────────────────────────
    eval_strategy="epoch",              # evaluate at end of every epoch
    save_strategy="epoch",              # save checkpoint at end of every epoch
    load_best_model_at_end=True,        # restore best checkpoint after training
    metric_for_best_model="f1",         # use F1 to pick the best checkpoint
    greater_is_better=True,

    # ── Logging ──────────────────────────────────────────────────────────────
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=50,
    report_to="none",                   # disable W&B / MLflow

    # ── Misc ─────────────────────────────────────────────────────────────────
    seed=SEED,
    fp16=torch.cuda.is_available(),     # mixed precision — speeds up T4 GPU training
    dataloader_num_workers=2,
)

print("TrainingArguments set.")
print(f"  fp16 (mixed precision): {training_args.fp16}")
print(f"  Effective batch size:   {BATCH_SIZE} per device")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=PATIENCE)
    ],
)

print("Trainer created.")
print(f"Training steps per epoch: {len(train_tok) // BATCH_SIZE}")
print(f"Total training steps:     {len(train_tok) // BATCH_SIZE * EPOCHS}")

In [ ]:
print("Starting training...")
print("-" * 60)

train_result = trainer.train()

print("\nTraining complete.")
print(f"Total training time: {train_result.metrics['train_runtime']:.1f}s")
print(f"Samples/second:      {train_result.metrics['train_samples_per_second']:.1f}")

In [ ]:
# Extract training history from Trainer logs
log_history = trainer.state.log_history

train_losses, val_losses   = [], []
val_accs, val_f1s          = [], []
epoch_nums                 = []

for entry in log_history:
    if "eval_loss" in entry:
        epoch_nums.append(entry["epoch"])
        val_losses.append(entry["eval_loss"])
        val_accs.append(entry.get("eval_accuracy", None))
        val_f1s.append(entry.get("eval_f1", None))
    if "loss" in entry and "eval_loss" not in entry:
        train_losses.append((entry["epoch"], entry["loss"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
if train_losses:
    t_epochs, t_loss = zip(*train_losses)
    ax1.plot(t_epochs, t_loss, label="Train Loss", alpha=0.6, color='steelblue')
ax1.plot(epoch_nums, val_losses, label="Val Loss", marker='s', color='tomato')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy & F1
ax2.plot(epoch_nums, val_accs, label="Val Accuracy", marker='o', color='steelblue')
ax2.plot(epoch_nums, val_f1s,  label="Val F1",       marker='s', color='darkorange')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Validation Accuracy & F1")
ax2.set_ylim(0.5, 1.0)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

if val_accs:
    print(f"Best val accuracy: {max(val_accs):.4f}")
    print(f"Best val F1:       {max(val_f1s):.4f}")

---
## 6. Evaluation

Evaluate the best checkpoint on the **test set** (held out during all training).

Metrics:
- **Accuracy** — overall fraction correct
- **Precision** — of predicted positives, how many are truly positive?
- **Recall** — of actual positives, how many did we catch?
- **F1** — harmonic mean of precision and recall
- **Confusion Matrix** — visualize error types

In [ ]:
print("Evaluating on test set...")
test_results = trainer.predict(test_tok)

logits     = test_results.predictions
all_labels = test_results.label_ids
all_preds  = np.argmax(logits, axis=-1)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)
f1   = f1_score(all_labels, all_preds, zero_division=0)
cm   = confusion_matrix(all_labels, all_preds)

print("\n" + "=" * 45)
print("TEST SET RESULTS")
print("=" * 45)
print(f"  Accuracy:  {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print("\nFull classification report:")
print(classification_report(all_labels, all_preds, target_names=["Negative", "Positive"]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'],
    ax=ax
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives:  {tp:,}")

In [ ]:
# ── Comparison with your custom Transformer ───────────────────────────────────
# Update these numbers with your actual Project A results
CUSTOM_ACCURACY = 0.819
CUSTOM_F1       = 0.809

print("=" * 55)
print("COMPARISON: Custom Transformer vs DistilRoBERTa")
print("=" * 55)
print(f"{'Model':<30} {'Accuracy':>10} {'F1':>10}")
print("-" * 55)
print(f"{'Your Custom Transformer':<30} {CUSTOM_ACCURACY:>10.1%} {CUSTOM_F1:>10.1%}")
print(f"{'DistilRoBERTa (fine-tuned)':<30} {acc:>10.1%} {f1:>10.1%}")
print("-" * 55)
print(f"{'Difference':<30} {acc - CUSTOM_ACCURACY:>+10.1%} {f1 - CUSTOM_F1:>+10.1%}")
print("=" * 55)
print("""
Why the gap?
  1. Pretraining data: DistilRoBERTa was trained on 160GB+ of text.
     Your model saw only IMDb (~70MB).
  2. Tokenization: BPE handles morphological variation better than
     a fixed word vocabulary.
  3. Scale: 82M pretrained parameters vs your architecture
     learning from scratch.
  4. Fine-tuning: we're adapting existing knowledge, not building it.
""")

---
## 7. Inference

A clean `predict()` function you can drop directly into a product.

In [ ]:
import torch.nn.functional as F

def predict(review: str, model, tokenizer, device, max_len=MAX_LEN):
    """
    Predict the sentiment of a single review.

    Args:
        review:    Raw review text (no preprocessing needed — tokenizer handles it)
        model:     Fine-tuned DistilRoBERTa model
        tokenizer: The matching tokenizer
        device:    torch.device
        max_len:   Max token length (default MAX_LEN)

    Returns:
        label:      "Positive" or "Negative"
        confidence: probability of the predicted class (0–100%)
        probs:      dict with both class probabilities
    """
    model.eval()

    encoding = tokenizer(
        review,
        max_length=max_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits                          # (1, 2)
        probs   = F.softmax(logits, dim=-1)[0].cpu()     # (2,)

    pred_class = probs.argmax().item()
    confidence = probs[pred_class].item() * 100
    label      = model.config.id2label[pred_class]

    return label, confidence, {
        "Negative": round(probs[0].item() * 100, 1),
        "Positive": round(probs[1].item() * 100, 1),
    }

In [ ]:
test_reviews = [
    "This movie was absolutely fantastic. The acting was superb and the plot kept me engaged throughout.",
    "Terrible film. Boring, predictable, and a complete waste of time. I want my money back.",
    "It was okay, nothing special. Some parts were good but overall pretty average.",
    "One of the best performances I've ever seen. A masterpiece of modern cinema.",
    "Dull, lifeless, and painfully slow. The dialogue was cringeworthy.",
]

print("INFERENCE RESULTS")
print("=" * 70)
for review in test_reviews:
    label, conf, all_probs = predict(review, model, tokenizer, DEVICE)
    short = review[:65] + "..." if len(review) > 65 else review
    print(f"Review:       {short}")
    print(f"Prediction:   {label} ({conf:.1f}%)")
    print(f"Probabilities: Negative={all_probs['Negative']}%  Positive={all_probs['Positive']}%")
    print("-" * 70)

In [ ]:
# Interactive cell — try your own review
my_review = "The cinematography was stunning but the story was deeply disappointing."

label, confidence, all_probs = predict(my_review, model, tokenizer, DEVICE)
print(f"Review:        {my_review}")
print(f"Prediction:    {label}")
print(f"Confidence:    {confidence:.1f}%")
print(f"Probabilities: Negative={all_probs['Negative']}%  Positive={all_probs['Positive']}%")

---
## 8. Save & Download Model

Two files are saved:
- `distilroberta-imdb/` — model weights + config (HuggingFace format)
- `distilroberta-imdb-tokenizer/` — tokenizer files

**Option A:** Download directly to your computer (browser download)  
**Option B:** Save to Google Drive (persists across Colab sessions)

In [ ]:
# ─── SAVE MODEL & TOKENIZER ──────────────────────────────────────────────────
SAVE_DIR           = "./distilroberta-imdb-final"
TOKENIZER_SAVE_DIR = "./distilroberta-imdb-tokenizer"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(TOKENIZER_SAVE_DIR)

print(f"Model saved to:     {SAVE_DIR}")
print(f"Tokenizer saved to: {TOKENIZER_SAVE_DIR}")
print(f"\nFiles in {SAVE_DIR}:")
for f in os.listdir(SAVE_DIR):
    size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"  {f:<35} {size_mb:.1f} MB")

In [ ]:
# ─── ZIP & DOWNLOAD (Option A) ───────────────────────────────────────────────
# Zips model + tokenizer into one file and downloads it to your computer.

import shutil

print("Zipping model and tokenizer...")
shutil.copytree(TOKENIZER_SAVE_DIR, os.path.join(SAVE_DIR, "tokenizer"), dirs_exist_ok=True)
zip_path = shutil.make_archive("distilroberta-imdb", "zip", SAVE_DIR)
print(f"Created: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")

try:
    from google.colab import files
    print("Downloading...")
    files.download(zip_path)
    print("Download started — check your browser downloads.")
except ImportError:
    print("Not running on Colab — file is saved at:", zip_path)

In [ ]:
# ─── SAVE TO GOOGLE DRIVE (Option B) ─────────────────────────────────────────
# Saves the model directory to MyDrive/distilroberta-imdb/
# Survives Colab session restarts — you won't need to retrain.

try:
    from google.colab import drive
    drive.mount("/content/drive")

    drive_dir = "/content/drive/MyDrive/distilroberta-imdb"
    shutil.copytree(SAVE_DIR, drive_dir, dirs_exist_ok=True)
    shutil.copytree(TOKENIZER_SAVE_DIR, os.path.join(drive_dir, "tokenizer"), dirs_exist_ok=True)

    print(f"Model saved to Google Drive: {drive_dir}")
    print(f"Files:")
    for f in os.listdir(drive_dir):
        print(f"  {f}")
except ImportError:
    print("Not running on Colab — skipping Drive mount.")

In [ ]:
# ─── HOW TO RELOAD IN YOUR APP ───────────────────────────────────────────────
# Copy this snippet into your FastAPI / Streamlit / Flask app.
# Point model_dir and tokenizer_dir to where you unzipped the download.

RELOAD_SNIPPET = '''
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# ── Load ──────────────────────────────────────────────────────────────────
model_dir     = "./distilroberta-imdb-final"        # path to unzipped model
tokenizer_dir = "./distilroberta-imdb-final/tokenizer"

tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

# ── Predict ───────────────────────────────────────────────────────────────
def predict(review: str):
    enc = tokenizer(review, max_length=512, truncation=True,
                    padding="max_length", return_tensors="pt")
    with torch.no_grad():
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device)
        ).logits
    probs      = F.softmax(logits, dim=-1)[0].cpu()
    pred_class = probs.argmax().item()
    label      = model.config.id2label[pred_class]
    confidence = probs[pred_class].item() * 100
    return label, confidence

# ── Example ───────────────────────────────────────────────────────────────
label, conf = predict("The movie was absolutely fantastic!")
print(f"{label} ({conf:.1f}%)")   # → Positive (98.3%)
'''

print(RELOAD_SNIPPET)

---
## What You've Built

| Component | What it does |
|---|---|
| `AutoTokenizer` | BPE tokenizer — handles subwords, punctuation, special tokens |
| `AutoModelForSequenceClassification` | DistilRoBERTa + 2-class linear head |
| `Trainer` | Training loop, LR schedule, checkpointing, early stopping |
| `compute_metrics` | Accuracy, precision, recall, F1 at each validation epoch |
| `predict()` | Production-ready single-review inference function |
| Save / Download | HuggingFace format — loadable in 4 lines anywhere |

**Next steps for a deployed product:**
- Wrap `predict()` in a FastAPI endpoint
- Or use Streamlit for a quick demo UI
- For lower latency: export to ONNX (`optimum` library) for CPU inference